# Titanic Example — Complete Missing-Value Handling

The Titanic dataset is perfect for practicing missing-value handling.

## Step 1: Import Libraries

In [89]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

## Step 2: Load Titanic Dataset

In [90]:
df = pd.read_csv("../../dataset/titanic/train.csv")

## Step 3: Inspect Dataset

In [91]:
print("Shape:", df.shape)

df.head()

Shape: (891, 12)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## Step 4: Detect Missing Values

In [92]:
missing_values = df.isnull().sum()

print(missing_values)

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64


## Step 5:  Calculate Missing-Value Percentage

In [93]:
missing_percentage = df.isnull().mean() * 100

print(missing_percentage)

PassengerId     0.000000
Survived        0.000000
Pclass          0.000000
Name            0.000000
Sex             0.000000
Age            19.865320
SibSp           0.000000
Parch           0.000000
Ticket          0.000000
Fare            0.000000
Cabin          77.104377
Embarked        0.224467
dtype: float64


### Note: If you want to show only columns where there are missing values:

In [94]:
missing_percentage = (
    df.isnull()
      .mean()
      .mul(100)
      .sort_values(ascending=False)
)

print(missing_percentage[missing_percentage > 0])

Cabin       77.104377
Age         19.865320
Embarked     0.224467
dtype: float64


## Step 6: Understand the Three Important Columns

In [95]:
print("Cabin missing:", df["Cabin"].isnull().sum())
print("Age missing:", df["Age"].isnull().sum())
print("Embarked missing:", df["Embarked"].isnull().sum())

Cabin missing: 687
Age missing: 177
Embarked missing: 2


## Step 7 — Age: Median Imputation

In [96]:
age_imputer = SimpleImputer(strategy="median")

df[["Age"]] = age_imputer.fit_transform(df[["Age"]])


print("Age missing after imputation:", df["Age"].isnull().sum())

Age missing after imputation: 0


## Step 8 — Embarked: Most Frequent / Mode Imputation

In [97]:
embarked_imputer = SimpleImputer(strategy="most_frequent")

df[["Embarked"]] = embarked_imputer.fit_transform(df[["Embarked"]])


print("Embarked missing after imputation:", df["Embarked"].isnull().sum())

Embarked missing after imputation: 0


## Step 9 — Cabin: Don't Blindly Impute

There are approximately 687 missing values ​​out of 891 in Cabin.

In [99]:
print("Cabin missing:",
      df["Cabin"].isnull().sum())

print("Cabin missing percentage:",
      df["Cabin"].isnull().mean()*100)

Cabin missing: 687
Cabin missing percentage: 77.10437710437711


#### With such a high missing percentage, we simply:

df["Cabin"].fillna(...)


will not do

#### Instead, for this basic missing-value exercise, we can drop the Cabin feature:

In [100]:
df = df.drop(columns=["Cabin"])

## Step 10 — Check All Missing Values

In [101]:
print(df.isnull().sum())

PassengerId    0
Survived       0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Embarked       0
dtype: int64


🔥 Important: Now the selected dataset does not contain missing values.

## Step 11 — Final Missing-Value Check

In [102]:
total_missing = df.isnull().sum().sum()
print("Total missing values:", total_missing)

Total missing values: 0


## Step 12 — Verify Dataset

In [103]:
print("Final shape:", df.shape)

df.head()

Final shape: (891, 11)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,S


## Step 13 — Data Leakage Prevention

This is a very important concept.

First, separate the features and the target:

In [104]:
X = df.drop(columns=["Survived"])
y = df["Survived"]

#### Now, train/test split:

In [105]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size = 0.2,
    random_state = 42
)

But there's an important issue here:

X still contains categorical columns (Sex, Embarked, etc.), so SimpleImputer shouldn't be used directly on all of X if we're doing proper numerical/categorical preprocessing.

To understand data leakage, consider a simple numerical example.

## Step 14 — Correct Imputation Workflow

Suppose we are using only age:

In [106]:
X = df[["Age"]].copy()
y = df["Survived"]

Split:

In [107]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size = 0.2,
    random_state = 42
)

Imputer:

In [108]:
imputer = SimpleImputer(strategy="median")

Correct:

In [109]:
imputer.fit(X_train)

X_train = imputer.transform(X_train)
X_test = imputer.transform(X_test)

#### Remember:
##### FIT       → Training data only
##### TRANSFORM → Training data
##### TRANSFORM → Test data

before train/test split when building a real ML model.

Because then test-set information can leak into the preprocessing process.

## Step 15 — Check Final Train/Test Data

In [110]:
print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

Training shape: (712, 1)
Testing shape: (179, 1)


## Step 16 — Complete Verification

In [111]:
print("Training missing values:",
      np.isnan(X_train).sum())

print("Testing missing values:",
      np.isnan(X_test).sum())

Training missing values: 0
Testing missing values: 0
